# 05 - Treinamento da CNN Propria

Este notebook implementa a tarefa 13 da ordem recomendada: treinar uma CNN propria criada para o CardioIA Vision.

A arquitetura esta em `src/models/custom_cnn.py` e usa blocos `Conv2D + BatchNorm + ReLU + MaxPooling`, seguidos de `Global Average Pooling` e camadas fully connected.

Saida do modelo: um logit por imagem. Loss recomendada: `BCEWithLogitsLoss`.

## Pre-requisitos

Execute antes:

1. `00_download_dataset_kaggle.ipynb`
2. `01_eda_dataset.ipynb`
3. `02_preprocessamento_splits.ipynb`
4. `03_validacao_preprocessamento_dataloaders.ipynb`
5. `04_validacao_pipeline_treino_avaliacao.ipynb`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
from torch import nn

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.datasets import DataLoaderConfig, create_dataloaders
from src.models.custom_cnn import create_custom_cnn
from src.training.evaluate import count_model_parameters, evaluate_model
from src.training.train import TrainConfig, train_model

config.ensure_project_directories()
config.seed_everything()

print("DEVICE:", config.DEVICE)
print("SPLITS_DIR:", config.SPLITS_DIR)

In [ ]:
required_splits = [config.SPLITS_DIR / f"{split}.csv" for split in ["train", "val", "test"]]
missing_splits = [path for path in required_splits if not path.exists()]

if missing_splits:
    raise FileNotFoundError(
        "Splits ausentes: "
        + ", ".join(str(path) for path in missing_splits)
        + ". Execute notebooks/02_preprocessamento_splits.ipynb primeiro."
    )

loader_config = DataLoaderConfig(
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    use_weighted_sampler=False,
)
loaders = create_dataloaders(dataloader_config=loader_config)

for split_name, loader in loaders.items():
    print(split_name, "imagens:", len(loader.dataset), "classes:", loader.dataset.class_counts)

## Criacao e teste de forward pass

Antes de treinar, validamos se a saida tem shape `[batch]`, compativel com `BCEWithLogitsLoss`.

In [ ]:
model = create_custom_cnn(dropout=0.35)
print(model)
print(count_model_parameters(model))

dummy_batch = torch.randn(4, 3, config.IMAGE_SIZE, config.IMAGE_SIZE)
with torch.no_grad():
    dummy_logits = model(dummy_batch)

print("Forward output shape:", tuple(dummy_logits.shape))

## Treinamento

Para um teste rapido, use poucas epocas. Para o experimento final, aumente `EPOCHS` conforme o desempenho no seu PC com GPU.

In [ ]:
EPOCHS = config.DEFAULT_EPOCHS
LEARNING_RATE = config.LEARNING_RATE

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

train_config = TrainConfig(
    model_name="cnn_propria_cardioia",
    epochs=EPOCHS,
    device=config.DEVICE,
    metric_to_maximize="f1",
    use_amp=torch.cuda.is_available(),
)

train_result = train_model(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    criterion=criterion,
    optimizer=optimizer,
    train_config=train_config,
)

train_result["summary"]

## Avaliacao no conjunto de teste

As metricas sao salvas em `reports/metricas/`, incluindo matriz de confusao e curva ROC.

In [ ]:
test_metrics = evaluate_model(
    model=model,
    dataloader=loaders["test"],
    model_name="cnn_propria_cardioia",
    device=config.DEVICE,
    output_dir=config.METRICS_DIR,
    checkpoint_path=train_result["best_checkpoint_path"],
)

test_metrics

## Proxima etapa

Treinar a CNN padrao de referencia em `06_treinamento_cnn_padrao.ipynb` usando os mesmos splits e metricas.